# 04 Candidate Evaluation

Track C merges outputs from both experimental views. The goal is a clean, report-friendly comparison table with shared scoring, method notes, and a single ranking rule.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

SEARCH_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in SEARCH_ROOTS if (path / "DAY2_data.txt").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sparsetap_config import DEFAULT_CONFIG
from sparsetap_utils import (
    candidate_table,
    load_candidates,
    prepare_environment,
    rank_candidates,
    save_candidates,
    save_table,
)

config = DEFAULT_CONFIG
data, prefix = prepare_environment(config)

predictive_candidates = load_candidates(config.candidate_dir / "predictive_candidates.json")
rule_candidates = load_candidates(config.candidate_dir / "rule_recovery_candidates.json")
all_candidates = predictive_candidates + rule_candidates

ranked = rank_candidates(all_candidates, data=data, prefix=prefix, noise=config.noise_prob)
save_candidates(ranked, config.candidate_dir / "all_ranked_candidates.json")
ranking_table = candidate_table(ranked)
save_table(ranking_table, config.metrics_dir / "all_candidate_ranking.csv")

by_method = (
    ranking_table.sort_values(
        ["prefix consistent?", "log-likelihood", "accuracy", "number of taps"],
        ascending=[False, False, False, True],
    )
    .groupby(["track", "method"], as_index=False)
    .head(1)
)
display(by_method)
ranking_table.head(40)
